In [2]:
import io

import numpy as np
import pandas as pd
import requests
import yfinance as yf

pd.set_option('display.width', 200)

HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
TODAY = pd.Timestamp('2026-08-21')          # fecha de corte del enunciado

In [3]:
#Q1
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
tables = pd.read_html(io.StringIO(requests.get(url, headers=HEADERS).text))
print('Tablas en la página:', len(tables), [list(t.columns) for t in tables])

# La tabla correcta es la que tiene 'Date added' (la página trae varias)
sp = next((t for t in tables if 'Date added' in t.columns), None)
if sp is None:
    raise KeyError('Ninguna tabla tiene "Date added". Ejecuta Kernel > Restart and Run All.')

sp = sp.copy()
sp.columns = [str(c).strip() for c in sp.columns]
sp.head()

Tablas en la página: 2 [['Symbol', 'Security', 'GICS Sector', 'GICS Sub-Industry', 'Headquarters Location', 'Date added', 'CIK', 'Founded'], ['vteS&P 500 companies', 'vteS&P 500 companies.1']]


,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989


In [4]:
sp['Year Added'] = pd.to_datetime(sp['Date added'], errors='coerce').dt.year

counts = sp['Year Added'].value_counts().sort_index()
recent = counts.loc[2020:]
recent.to_frame('adiciones')

,adiciones
Year Added,
2020,10
2021,10
2022,15
2023,14
2024,16
2025,17
2026,16


In [5]:
print('Año con más incorporaciones desde 2020:', int(recent.idxmax()))
print('Adiciones ese año:', int(recent.max()))
print()
print('Adicional: acciones en el índice hace más de 20 años (Year Added < 2006):',
      int((sp['Year Added'] < 2006).sum()), 'de', len(sp))

Año con más incorporaciones desde 2020: 2025
Adiciones ese año: 17

Adicional: acciones en el índice hace más de 20 años (Year Added < 2006): 217 de 503


In [6]:
#Q2
indices = {
    'US S&P 500': '^GSPC',
    'China Shanghai': '000001.SS',
    'HK Hang Seng': '^HSI',
    'Australia ASX200': '^AXJO',
    'India Nifty 50': '^NSEI',
    'Canada TSX': '^GSPTSE',
    'Germany DAX': '^GDAXI',
    'UK FTSE100': '^FTSE',
    'Japan Nikkei225': '^N225',
    'Mexico IPC': '^MXX',
    'Brazil Ibovespa': '^BVSP',
}
list(indices.items())

[('US S&P 500', '^GSPC'),
 ('China Shanghai', '000001.SS'),
 ('HK Hang Seng', '^HSI'),
 ('Australia ASX200', '^AXJO'),
 ('India Nifty 50', '^NSEI'),
 ('Canada TSX', '^GSPTSE'),
 ('Germany DAX', '^GDAXI'),
 ('UK FTSE100', '^FTSE'),
 ('Japan Nikkei225', '^N225'),
 ('Mexico IPC', '^MXX'),
 ('Brazil Ibovespa', '^BVSP')]

In [7]:
ytd = {}
for name, ticker in indices.items():
    data = yf.download(ticker, start='2025-11-01', end='2026-08-22',
                       progress=False, auto_adjust=False)
    close = data['Close']
    if isinstance(close, pd.DataFrame):          # yfinance moderno -> MultiIndex
        close = close.squeeze(axis=1)
    close = close.dropna()
    base = float(close[close.index <= '2025-12-31'].iloc[-1])
    end = float(close[close.index <= '2026-08-21'].iloc[-1])
    ytd[name] = (end / base - 1) * 100

ytd_df = pd.Series(ytd, name='YTD %').sort_values(ascending=False).to_frame()
ytd_df.round(2)

us = ytd['US S&P 500']
better = [k for k, v in ytd.items() if k != 'US S&P 500' and v > us]

print('S&P 500 YTD: %+.2f %%' % us)
print('Índices que lo baten:', len(better), better)
print()
print('RESPUESTA:', len(better))

S&P 500 YTD: +12.11 %
Índices que lo baten: 2 ['Canada TSX', 'Japan Nikkei225']

RESPUESTA: 2


In [8]:
for years in [1, 3, 5, 10]:
    start = TODAY - pd.DateOffset(years=years)
    rets = {}
    for name, ticker in indices.items():
        data = yf.download(ticker, start=start.strftime('%Y-%m-%d'), end='2026-08-22',
                           progress=False, auto_adjust=True)
        close = data['Close']
        if isinstance(close, pd.DataFrame):
            close = close.squeeze(axis=1)
        close = close.dropna()
        if len(close) >= 2:
            rets[name] = (close.iloc[-1] / close.iloc[0] - 1) * 100
    us_long = rets.get('US S&P 500')
    winners = sorted(k for k, v in rets.items() if k != 'US S&P 500' and v > us_long)
    print('%2d años -> %d índices (%s) | S&P 500 %+.2f %%'
          % (years, len(winners), ', '.join(winners), us_long))

 1 años -> 3 índices (Brazil Ibovespa, Canada TSX, Japan Nikkei225) | S&P 500 +20.47 %
 3 años -> 2 índices (Canada TSX, Japan Nikkei225) | S&P 500 +74.43 %
 5 años -> 2 índices (Canada TSX, Japan Nikkei225) | S&P 500 +71.32 %
10 años -> 1 índices (Japan Nikkei225) | S&P 500 +251.61 %


In [9]:
#Q3
sp500 = yf.download('^GSPC', start='1950-01-01', progress=False, auto_adjust=False)
close = sp500['Close']
if isinstance(close, pd.DataFrame):
    close = close.squeeze(axis=1)
close = close.dropna()

print('Rango:', close.index[0].date(), '->', close.index[-1].date(), '|', len(close), 'sesiones')


Rango: 1950-01-03 -> 2026-09-28 | 19306 sesiones


In [10]:
peaks = close[close == close.cummax()]
peak_list = list(zip(peaks.index, peaks.values))
print('Máximos históricos:', len(peak_list), '| último:', peak_list[-1][0].date())
rows = []
for i in range(len(peak_list) - 1):
    peak_date, peak_value = peak_list[i]
    next_peak_date = peak_list[i + 1][0]
    window = close[(close.index > peak_date) & (close.index <= next_peak_date)]
    if window.empty:
        continue
    trough_value = window.min()
    trough_date = window.idxmin()
    drawdown = (peak_value - trough_value) / peak_value * 100
    if drawdown >= 5.0:
        trading_days = len(close[(close.index >= peak_date) & (close.index <= trough_date)]) - 1
        rows.append({
            'peak': str(peak_date.date()),
            'trough': str(trough_date.date()),
            'drawdown_pct': round(drawdown, 1),
            'dias_habiles': int(trading_days),
            'dias_naturales': int((trough_date - peak_date).days),
        })

corrections = pd.DataFrame(rows)
print('Correcciones >= 5 %:', len(corrections))
corrections.head()

Máximos históricos: 1537 | último: 2026-08-13
Correcciones >= 5 %: 74


,peak,trough,drawdown_pct,dias_habiles,dias_naturales
0,1950-06-12,1950-07-17,14.0,24,35
1,1950-11-24,1950-12-04,6.5,6,10
2,1951-05-03,1951-06-29,8.1,40,57
3,1951-10-15,1951-11-23,6.1,26,39
4,1952-01-22,1952-02-20,6.4,20,29


In [11]:
summary = {
    'drawdown_pct': {
        'p25': round(corrections['drawdown_pct'].quantile(.25), 1),
        'mediana': round(corrections['drawdown_pct'].median(), 1),
        'p75': round(corrections['drawdown_pct'].quantile(.75), 1),
    },
    'dias_habiles': {
        'p25': int(corrections['dias_habiles'].quantile(.25)),
        'mediana': int(corrections['dias_habiles'].median()),
        'p75': int(corrections['dias_habiles'].quantile(.75)),
    },
    'dias_naturales': {
        'p25': int(corrections['dias_naturales'].quantile(.25)),
        'mediana': int(corrections['dias_naturales'].median()),
        'p75': int(corrections['dias_naturales'].quantile(.75)),
    },
}
print(pd.DataFrame(summary).to_string())

print()
print('RESPUESTA (mediana del drawdown): %.1f %%' % corrections['drawdown_pct'].median())
corrections.nlargest(10, 'drawdown_pct').to_string(index=False)

         drawdown_pct  dias_habiles  dias_naturales
p25               6.2            14              22
mediana           8.0            27              40
p75              14.0            61              86

RESPUESTA (mediana del drawdown): 8.0 %


'      peak     trough  drawdown_pct  dias_habiles  dias_naturales\n2007-10-09 2009-03-09          56.8           355             517\n2000-03-24 2002-10-09          49.1           637             929\n1973-01-11 1974-10-03          48.2           436             630\n1968-11-29 1970-05-26          36.1           369             543\n2020-02-19 2020-03-23          33.9            23              33\n1987-08-25 1987-12-04          33.5            71             101\n1961-12-12 1962-06-26          28.0           135             196\n1980-11-28 1982-08-12          27.1           430             622\n2022-01-03 2022-10-12          25.4           195             282\n1966-02-09 1966-10-07          22.2           167             240'

In [12]:
#Q4
earnings = yf.Ticker('AMZN').get_earnings_dates().sort_index()
print('Filas:', len(earnings))
print('Columnas:', list(earnings.columns))
earnings.head()
earnings = earnings.dropna(subset=['Reported EPS'])
earnings['Surprise(%)'] = pd.to_numeric(earnings['Surprise(%)'], errors='coerce')
positive = earnings[earnings['Surprise(%)'] > 0]

print('Trimestres con EPS reportado:', len(earnings))
print('Sorpresas positivas:', len(positive))
positive.tail()

amzn = yf.download('AMZN', period='max', progress=False, auto_adjust=True)
close = amzn['Close']
if isinstance(close, pd.DataFrame):
    close = close.squeeze(axis=1)
close = close.dropna()

position_of = {d.date(): i for i, d in enumerate(close.index)}
print('Sesiones:', len(close), '|', close.index[0].date(), '->', close.index[-1].date())

rows = []
for date, row in positive.iterrows():
    day = date.date() if hasattr(date, 'date') else pd.Timestamp(date).date()
    i = position_of.get(day)
    if i is None or i < 1 or i + 1 >= len(close):
        continue                                  # falta día 1 o día 3 alrededor
    rows.append({
        'fecha': str(day),
        'sorpresa_pct': float(row['Surprise(%)']),
        'ret2d_pct': round(float((close.iloc[i + 1] / close.iloc[i - 1] - 1) * 100), 3),
    })

reactions = pd.DataFrame(rows)
reactions

print('Mediana del cambio a 2 días: %+.2f %%' % reactions['ret2d_pct'].median())
print('Media del cambio a 2 días:   %+.2f %%' % reactions['ret2d_pct'].mean())
print('Correlación sorpresa vs retorno a 2 días: %+.2f'
      % reactions[['sorpresa_pct', 'ret2d_pct']].corr().iloc[0, 1])
print()
print('RESPUESTA (mediana):', round(reactions['ret2d_pct'].median(), 2), '%')

Filas: 25
Columnas: ['EPS Estimate', 'Reported EPS', 'Surprise(%)']
Trimestres con EPS reportado: 24
Sorpresas positivas: 20
Sesiones: 7388 | 1997-05-15 -> 2026-09-28
Mediana del cambio a 2 días: +0.35 %
Media del cambio a 2 días:   +0.46 %
Correlación sorpresa vs retorno a 2 días: +0.33

RESPUESTA (mediana): 0.35 %


In [13]:
sp = yf.download('^GSPC', period='max', progress=False, auto_adjust=True)
sp_close = sp['Close']
if isinstance(sp_close, pd.DataFrame):
    sp_close = sp_close.squeeze(axis=1)
sp_close = sp_close.dropna()
sp_momentum = sp_close / sp_close.shift(252) - 1

dates = pd.to_datetime(reactions['fecha'])
regime = np.where(sp_momentum.reindex(dates).to_numpy() > 0, 'alcista', 'bajista')
reactions['regimen'] = regime

reactions.groupby('regimen')['ret2d_pct'].agg(['count', 'mean', 'median']).round(2)

,count,mean,median
regimen,,,
alcista,18,1.08,1.16
bajista,2,-5.07,-5.07
